# Create, evaluate, and deploy an AI agent

This notebook walks you through building, evaluating, and deploying an AI agent that combines retrieval and tool usage. You'll work with a pre-chunked subset of Databricks documentation as your dataset.


In [0]:
%pip install -U -qqqq mlflow langchain langgraph==0.3.4 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks]
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Get the catalog and schema name from the Databricks Unity Catalog, you will need this in later steps.

# Create an agent and tools

In [0]:
from databricks_langchain import ChatDatabricks

# TODO: Replace with your model serving endpoint
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

In [0]:
import pandas as pd

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)

In [0]:
from databricks_langchain.uc_ai import (
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)

uc_client = DatabricksFunctionClient()
set_uc_function_client(uc_client)


def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    def extract_keywords(text, top_n=5):
        """Extracts top keywords from input text using trained TF-IDF vectorizer"""
        keyword_vectorizer = TfidfVectorizer(
            stop_words="english"
        )  # New vectorizer for query
        query_tfidf = keyword_vectorizer.fit_transform([text])  # Fit on query only
        scores = query_tfidf.toarray()[0]
        indices = scores.argsort()[-top_n:][::-1]  # Get top N keywords
        return [
            keyword_vectorizer.get_feature_names_out()[i]
            for i in indices
            if scores[i] > 0
        ]

    return extract_keywords(text)


# TODO fill in your catalog and schema name
catalog = "test_databricks_ak"
schema = "default"

assert (catalog and schema)

# Create the function within the Unity Catalog catalog and schema specified
function_info = uc_client.create_python_function(
    func=tfidf_keywords,
    catalog=catalog,
    schema=schema,
    replace=True,  # Set to True to overwrite if the function already exists
)

uc_tool_names = [f"{catalog}.{schema}.tfidf_keywords"]
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)

/home/spark-3c2dcbee-47cb-4bf0-848f-cb/.ipykernel/300208/command-7851625625060281-342368373:1: DeprecationWarning: Imports from this module are deprecated and will be removed in a future release. Please update the code to import directly from databricks_langchain.

For example, replace imports like: `from databricks_langchain.uc_ai import UCFunctionToolkit`
with: `from databricks_langchain import UCFunctionToolkit`
  from databricks_langchain.uc_ai import (
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


In [0]:
print(uc_toolkit.tools[0])
uc_toolkit.tools[0].invoke({"text": "The quick brown fox jumped over the lazy brown dog."})

name='test_databricks_ak__default__tfidf_keywords' description='Extracts keywords from the provided text using TF-IDF.' args_schema=<class 'unitycatalog.ai.core.utils.function_processing_utils.test_databricks_ak__default__tfidf_keywords__params'> func=<function UCFunctionToolkit.uc_function_to_langchain_tool.<locals>.func at 0x7f52a7ee7d90> uc_function_name='test_databricks_ak.default.tfidf_keywords' client_config={'profile': None}


'{"format": "SCALAR", "value": "[\'brown\', \'quick\', \'lazy\', \'jumped\', \'fox\']"}'

In [0]:
from typing import Any

import mlflow
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result


In [0]:
from typing import Optional, Sequence, Union

from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()

In [0]:
import mlflow

mlflow.langchain.autolog()

agent = create_tool_calling_agent(llm, tools=[*uc_toolkit.tools, find_relevant_documents])
agent.invoke({"messages": [{"role": "user", "content":"What are the keywords for the sentence: 'the quick brown fox jumped over the lazy brown dog'?"}]})

{'messages': [{'role': 'user',
   'content': "What are the keywords for the sentence: 'the quick brown fox jumped over the lazy brown dog'?",
   'id': 'b2d82f42-9fef-4c15-a58a-130a7328fe86'},
  {'role': 'assistant',
   'content': '',
   'id': 'run--019c1f7a-9dde-7cd2-b8de-fbbf5ac05c24-0',
   'tool_calls': [{'id': 'call_942452fa-9868-4531-afd0-96986d2a6720',
     'type': 'function',
     'function': {'name': 'test_databricks_ak__default__tfidf_keywords',
      'arguments': '{"text": "the quick brown fox jumped over the lazy brown dog"}'},
     'name': 'test_databricks_ak__default__tfidf_keywords',
     'args': {'text': 'the quick brown fox jumped over the lazy brown dog'}}]},
  {'role': 'tool',
   'content': '{"format": "SCALAR", "value": "[\'brown\', \'quick\', \'lazy\', \'jumped\', \'fox\']"}',
   'name': 'test_databricks_ak__default__tfidf_keywords',
   'id': 'cbc134ca-faea-4193-b29a-5a87b812843d',
   'tool_call_id': 'call_942452fa-9868-4531-afd0-96986d2a6720'},
  {'role': 'assistant

Trace(trace_id=tr-12551940ae6b6240085997fa161a8cdb)

In [0]:
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
from typing import Any, Optional

class DocsAgent(ChatAgent):
  def __init__(self, agent):
    self.agent = agent

  def predict(
      self,
      messages: list[ChatAgentMessage],
      context: Optional[ChatContext] = None,
      custom_inputs: Optional[dict[str, Any]] = None,
  ) -> ChatAgentResponse:
      # ChatAgent has a built-in helper method to help convert framework-specific messages, like langchain BaseMessage to a python dictionary
      request = {"messages": self._convert_messages_to_dict(messages)}

      output = agent.invoke(request)
      # Here 'output' is already a ChatAgentResponse, but to make the ChatAgent signature explicit for this demonstration we are returning a new instance
      return ChatAgentResponse(**output)

In [0]:
AGENT = DocsAgent(agent=agent)
AGENT.predict({"messages": [{"role": "user", "content": "What is DLT in Databricks?"}]})

ChatAgentResponse(messages=[ChatAgentMessage(role='user', content='What is DLT in Databricks?', name=None, id='d172e10a-78e5-4d3f-9f4f-9593438693bf', tool_calls=None, tool_call_id=None, attachments=None), ChatAgentMessage(role='assistant', content='DLT stands for Delta Live Tables, which is a feature in Databricks that allows users to build and manage data pipelines in a scalable and reliable way. It provides a simple and intuitive way to define data transformations and load data into Delta Lake tables, which are optimized for performance and reliability. With DLT, users can define pipelines using a simple and declarative syntax, and Databricks will automatically manage the execution of the pipeline, including handling failures, retries, and optimization. This allows users to focus on defining the data transformations and loading data, rather than worrying about the underlying infrastructure and execution details.', name=None, id='run--019c1f7b-50f4-7492-9199-c8ffb906b75b-0', tool_call

Trace(trace_id=tr-4f01f69b818f660a0f3bb6d3edbbc11f)

In [0]:
from mlflow.models import ModelConfig

baseline_config = {
   "endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
   "temperature": 0.01,
   "max_tokens": 1000,
   "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

    You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
    """,
   "tool_list": [f"{catalog}.{schema}.*"],
}


class DocsAgent(ChatAgent):
    def __init__(self):
        self.config = ModelConfig(development_config=baseline_config)
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        temperature = self.config.get("temperature")
        max_tokens = self.config.get("max_tokens")
        system_prompt = self.config.get("system_prompt")
        llm_endpoint_name = self.config.get("endpoint_name")
        tool_list = self.config.get("tool_list")

        llm = ChatDatabricks(endpoint=llm_endpoint_name, temperature=temperature, max_tokens=max_tokens)
        toolkit = UCFunctionToolkit(function_names=tool_list)
        agent = create_tool_calling_agent(llm, tools=[*toolkit.tools, find_relevant_documents], agent_prompt=system_prompt)

        return agent
    
    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper method to help convert framework-specific messages, like langchain BaseMessage to a python dictionary
        request = {"messages": self._convert_messages_to_dict(messages)}

        output = self.agent.invoke(request)
        # Here 'output' is already a ChatAgentResponse, but to make the ChatAgent signature explicit for this demonstration we are returning a new instance
        return ChatAgentResponse(**output)

agent = DocsAgent()
agent.predict({"messages": [{"role": "user", "content": "What is DLT"}]})

ChatAgentResponse(messages=[ChatAgentMessage(role='user', content='What is DLT', name=None, id='08e16332-db67-4831-8b5a-24896042bfa0', tool_calls=None, tool_call_id=None, attachments=None), ChatAgentMessage(role='assistant', content='DLT stands for Delta Live Tables, which is a feature in Databricks that allows users to build and manage data pipelines in a scalable and reliable way. It provides a simple and intuitive way to define data transformations and loading processes using a SQL-like syntax, making it easier to manage complex data workflows. DLT enables users to create and manage data pipelines that can handle large volumes of data, perform data validation and quality checks, and provide real-time monitoring and alerts. It is designed to simplify the process of building and managing data pipelines, making it easier to get insights from data and make data-driven decisions.', name=None, id='run--019c1f7b-c836-7993-b395-560fc200f0e8-0', tool_calls=None, tool_call_id=None, attachment

Trace(trace_id=tr-c55e77fa7edb9754e30a9831d07b9d02)

In [0]:
%%writefile getting_started_agent.py
from typing import Any, Optional, Sequence, Union

import mlflow
import pandas as pd
from databricks_langchain import ChatDatabricks
from databricks_langchain.uc_ai import (
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.models import ModelConfig
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
from sklearn.feature_extraction.text import TfidfVectorizer

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class DocsAgent(ChatAgent):
    def __init__(self, config, tools):
        # Load config
        # When this agent is deployed to Model Serving, the configuration loaded here is replaced with the config passed to mlflow.pyfunc.log_model(model_config=...)
        self.config = ModelConfig(development_config=config)
        self.tools = tools
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        llm = ChatDatabricks(
            endpoint=self.config.get("endpoint_name"),
            temperature=self.config.get("temperature"),
            max_tokens=self.config.get("max_tokens"),
        )
        agent = create_tool_calling_agent(
            llm,
            tools=self.tools,
            agent_prompt=self.config.get("system_prompt"),
        )
        return agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper method to help convert framework-specific messages, like langchain BaseMessage to a python dictionary
        request = {"messages": self._convert_messages_to_dict(messages)}

        output = self.agent.invoke(request)
        # Here 'output' is already a ChatAgentResponse, but to make the ChatAgent signature explicit for this demonstration we are returning a new instance
        return ChatAgentResponse(**output)
    

# TODO fill in your catalog and schema name
catalog = "test_databricks_ak"
schema = "default"

# TODO: Replace with your model serving endpoint
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

baseline_config = {
    "endpoint_name": LLM_ENDPOINT,
    "temperature": 0.01,
    "max_tokens": 1000,
    "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

    You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
    """,
}

tools = [find_relevant_documents]
uc_client = DatabricksFunctionClient()
set_uc_function_client(uc_client)
uc_toolkit = UCFunctionToolkit(function_names=[f"{catalog}.{schema}.*"])
tools.extend(uc_toolkit.tools)


AGENT = DocsAgent(baseline_config, tools)
mlflow.models.set_model(AGENT)

Overwriting getting_started_agent.py


In [0]:
dbutils.library.restartPython()

In [0]:
from getting_started_agent import AGENT

AGENT.predict({"messages": [{"role": "user", "content": "What is DLT"}]})

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


ChatAgentResponse(messages=[ChatAgentMessage(role='user', content='What is DLT', name=None, id='b7038c30-ff64-4fe3-a786-46e592b23840', tool_calls=None, tool_call_id=None, attachments=None), ChatAgentMessage(role='assistant', content='DLT stands for Delta Live Tables, which is a feature in Databricks that allows users to build and manage data pipelines in a scalable and reliable way. It provides a simple and intuitive way to define data transformations and load data into Delta Lake tables. With DLT, users can define pipelines using a SQL-like syntax, and Databricks will automatically manage the execution of the pipeline, including handling failures, retries, and optimization. DLT also provides features such as data quality checks, data validation, and data lineage, making it a powerful tool for building and managing data pipelines in the cloud.', name=None, id='run--019c1f7c-f226-7332-a7dc-848355a7fa41-0', tool_calls=None, tool_call_id=None, attachments=None)], finish_reason=None, custo

In [0]:
import mlflow
from getting_started_agent import LLM_ENDPOINT, baseline_config, tools
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT)]
for tool in tools:
    if isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        python_model="getting_started_agent.py",
        artifact_path="agent",
        model_config=baseline_config,
        resources=resources,
        pip_requirements=[
            "mlflow",
            "langchain",
            "langgraph==0.3.4",
            "databricks-langchain",
            "unitycatalog-langchain[databricks]",
            "pydantic",
        ],
        input_example={
            "messages": [{"role": "user", "content": "What is lakehouse monitoring?"}]
        },
    )

2026/02/02 17:54:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405619831884426.6.azuredatabricks.net/ml/experiments/4342094710673056/models/m-08ea0143a99549969e652168940ef856?o=7405619831884426
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)
2026/02/02 17:54:08 INFO mlflow.pyfunc: Predicting on input example to validate output
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect s

In [0]:
import pandas as pd
from databricks.agents.evals import generate_evals_df

agent_description = """
The agent is a RAG chatbot that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.
"""
question_guidelines = """
# User personas
- A developer who is new to the Databricks platform
- An experienced, highly technical Data Scientist or Data Engineer


# Example questions
- what API lets me parallelize operations over rows of a delta table?
- Which cluster settings will give me the best performance when using Spark?


# Additional Guidelines
- Questions should be succinct, and human-like
"""


databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)


num_evals = 25
evals = generate_evals_df(
    docs=parsed_docs_df[
        :500
    ],  # Pass your docs. They should be in a Pandas or Spark DataFrame with columns `content STRING` and `doc_uri STRING`.
    num_evals=num_evals,  # How many synthetic evaluations to generate
    agent_description=agent_description,
    question_guidelines=question_guidelines,
)
display(evals)

Generating evaluations:   0%|          | 0/25 evals generated [Elapsed: 00:00, Remaining: ?]

request_id,source_type,source_id,inputs,expectations
06ccee3f2c9480c9f365d66f7faf7a26068e946149f888b799d168bde3d23107,SYNTHETIC_FROM_DOC,https://docs.databricks.com/delta-live-tables/python-ref.html,"List(List(List(How do you access an internal dataset using a spark.sql expression in Delta Live Tables?, user)))","List(List(The dataset name should be prefixed with `LIVE.` to access it using a spark.sql expression in Delta Live Tables, Correct syntax for accessing the dataset involves using `spark.sql(""SELECT * FROM LIVE."")`), List(List(You can also return a dataset using a `spark.sql` expression in a query function. To read from an internal dataset, prepend `LIVE.` to the dataset name: ``` @dlt.table def chicago_customers(): return spark.sql(""SELECT * FROM LIVE.customers_cleaned WHERE city = 'Chicago'"") ```, https://docs.databricks.com/delta-live-tables/python-ref.html)))"
8cabb03b8652be40ef763967b3078df8d5bcbf8ab1fa37bea908ba9d426da1b0,SYNTHETIC_FROM_DOC,https://docs.databricks.com/security/keys/customer-managed-keys.html,"List(List(List(Which parts of the serverless compute resources in Databricks do not support customer-managed keys for EBS storage?, user)))","List(List(Compute nodes within serverless compute resources do not support customer-managed keys for EBS storage., Disks associated with serverless compute resources do not support customer-managed keys for EBS storage.), List(List(Serverless SQL warehouses do not use customer-managed keys for EBS storage encryption on compute nodes, which is an optional part of configuring customer-managed keys for workspace storage. Disks for serverless compute resources are short-lived and tied to the lifecycle of the serverless workload. When compute resources are stopped or scaled down, the VMs and their storage are destroyed."" ""Customer-managed keys for EBS storage, which is an optional part of the customer-managed workspace storage feature, does *not* apply to serverless compute resources. Disks for serverless compute resources are short-lived and tied to the lifecycle of the serverless workload. When compute resources are stopped or scaled down, the VMs and their storage are destroyed., https://docs.databricks.com/security/keys/customer-managed-keys.html)))"
04fe1f40f3af12afe447e01e1f0ea9ab26501a2c4a9dfc74515afb450630eed1,SYNTHETIC_FROM_DOC,https://docs.databricks.com/repos/errors-troubleshooting.html,"List(List(List(How can I resolve notebook name conflicts when using Git integration in Databricks?, user)))","List(List(Rename notebooks, files, or folders causing the error., Ensure each item has a unique name., Rename the items in the remote Git repository if the conflict occurs when cloning the repo.), List(List(Different notebooks with identical or similar filenames can cause an error when you create a repo or pull request, such as `Cannot perform Git operation due to conflicting names` or `A folder cannot contain a notebook with the same name as a notebook, file, or folder (excluding file extensions).` A naming conflict can occur even with different file extensions. For example, these two files conflict: * `notebook.ipynb` * `notebook.py` ### To fix the name conflict * Rename the notebook, file, or folder contributing to the error state. + If this error occurs when you clone the repo, you need to rename notebooks, files, or folders in the remote Git repo., https://docs.databricks.com/repos/errors-troubleshooting.html)))"
db1e57371d95ecdafb1afafdbb235ef6aa5e4c94b6d569ab832c5cb545a1a99b,SYNTHETIC_FROM_DOC,https://docs.databricks.com/discover/databricks-datasets.html,"List(List(List(How can I browse Databricks datasets from a Python, Scala, or R notebook?, user)))","List(List(Usage of a command or method to browse Databricks datasets., The `dbutils.fs.ls('/databricks-datasets')` command specifically used for Python (or equivalent command in Scala/R)., Command intended to list the contents of the '/databricks-datasets' directory.), List(List(To browse these files from a Py

In [0]:
from databricks.agents.evals import metric
from getting_started_agent import catalog, schema
@metric
def uses_keywords_and_retriever(request, trace):
  retriever_spans = trace.search_spans(span_type='RETRIEVER')
  keyword_tool_spans = trace.search_spans(name=f"{catalog}__{schema}__tfidf_keywords")
  return len(keyword_tool_spans) > 0 and len(retriever_spans) > 0

In [0]:
#This cell throws rate limit error. Please use a provisioned throughput Foundation Model APIs endpoint for a higher rate limit.

with mlflow.start_run(run_name="my_agent"):
  eval_results = mlflow.evaluate(
      data=evals,  # Your evaluation set
      model=model_info.model_uri,  # Logged agent from above
      model_type="databricks-agent",  # activate Mosaic AI Agent Evaluation,
      extra_metrics=[uses_keywords_and_retriever]
  )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/mlflow/models/evaluation/deprecated.py:9: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(


/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/connect/session.py:451: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)
2026/02/02 17:55:18 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-08ea0143a99549969e652168940ef856
2026/02/02 17:55:18 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/02/02 17:55:19 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/25 [Elapsed: 00:00, Remaining: ?] 

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/rag_eval/evaluation/metrics.py:131: FutureWarning: ``mlflow.metrics.recall_at_k`` is deprecated since 3.4.0. Use the new GenAI evaluation functionality instead. See https://mlflow.org/docs/latest/genai/eval-monitor/legacy-llm-evaluation/ for the migration guide.
  mlflow_eval_metric = getattr(mlflow.metrics, f"{metric_name}_at_k")(k)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/databricks/rag_eval/mlflow/databricks_rag_evaluator.py:158: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  search_traces_df = mlflow.search_traces(experiment_ids=[experiment_id], run_id=run_id)


<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-91f87caa5db85bec358b5fe1f0032e17), Trace(trace_id=tr-55be61897291a47927247836eb45222e), Trace(trace_id=tr-5e3c67447255c5824baa9192ae57085b), Trace(trace_id=tr-d50c8f41a6d5470d282fcf693ce8253d), Trace(trace_id=tr-b571119cc8c0255ff6371b405d9b8111), Trace(trace_id=tr-f4079f9edc9999a810649cd81ede7d48), Trace(trace_id=tr-8ebc13dd0bb010bddc2670bf475cfa4a), Trace(trace_id=tr-8248c5b41fb57363709938a3a547631b), Trace(trace_id=tr-0f135f8a28038acebd306ee230f503f1), Trace(trace_id=tr-9d067765c89109ccedf2ff279de1e471)]

In [0]:
import mlflow
from databricks import agents

# Connect to the Unity catalog model registry
mlflow.set_registry_uri("databricks-uc")


# TODO: define the catalog and schema for your UC model
catalog = "test_databricks_ak"
schema = "default"
assert (catalog and schema)
UC_MODEL_NAME = f"{catalog}.{schema}.getting_started_agent"


# Register to Unity catalog
uc_registered_model_info = mlflow.register_model(
    model_uri=model_info.model_uri, name=UC_MODEL_NAME
)
# Deploy to enable the review app and create an API endpoint
deployment_info = agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, deploy_feedback_model=False)

Successfully registered model 'test_databricks_ak.default.getting_started_agent'.


---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/mlflow/utils/_unity_catalog_utils.py:194, in get_artifact_repo_from_storage_info(storage_location, scoped_token, base_credential_refresh_def, is_oss)
    193     else:
--> 194         return _get_artifact_repo_from_storage_info(
    195             storage_location=storage_location,
    196             scoped_token=scoped_token,
    197             base_credential_refresh_def=base_credential_refresh_def,
    198         )
    199 except ImportError as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-3c2dcbee-47cb-4bf0-848f-cbec461336e9/lib/python3.10/site-packages/mlflow/utils/_unity_catalog_utils.py:243, in _get_artifact_repo_from_storage_info(storage_location, scoped_token, base_credential_refresh_def)
    242 elif cred